# Retrieval Baselines

## Objective

The requirement extraction stage converts unstructured job descriptions into
structured requirements.

The next stage is candidate retrieval: given a job description, retrieve the
most relevant resumes from a large candidate pool.

This notebook establishes progressively stronger retrieval baselines:

1. TF-IDF
2. BM25
3. Dense semantic retrieval
4. Sparse and hybrid retrieval

The retrieval systems will be evaluated using:

- Recall@1
- Recall@3
- Recall@5
- Recall@10
- Mean Reciprocal Rank (MRR)
- nDCG@5
- nDCG@10

The candidate-matching dataset provides 2,500 jobs, 10,000 resumes, and
75,000 job-resume relevance relationships. Each job has 30 known relevant
resumes.

The purpose of this stage is to establish reproducible retrieval baselines
before introducing reranking and requirement-level evidence matching.

## Retrieval Problem

For each job \(J\), we have a candidate pool containing thousands of resumes.

The retrieval system produces a ranked list:

\[
J \rightarrow [R_1, R_2, R_3, \ldots, R_k]
\]

where \(R_i\) represents a resume.

The ground-truth relevance relationships allow us to evaluate whether
relevant candidates appear near the top of the ranking.

This stage evaluates **candidate retrieval**, not requirement-level evidence
classification.

The distinction is important:

- Retrieval asks: *Which resumes are relevant to this job?*
- Evidence matching asks: *Which resume evidence satisfies each specific job
  requirement?*

The latter will be addressed in later stages.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

print("Environment ready.")

Environment ready.


## 1. Dataset Loading

The retrieval dataset contains three logical components:

- resumes
- jobs
- job-resume relevance relationships

The relevance file provides the ground-truth candidate IDs associated with
each job.

In [2]:
from datasets import load_dataset

print("Dataset loader ready.")

Dataset loader ready.


## 2. Load the Candidate-Matching Dataset

The dataset is stored as separate Parquet files for resumes, jobs, and
job-resume relevance relationships.

The Hugging Face dataset repository exposes these files directly, so we load
the Parquet files rather than relying on automatic dataset configuration
inference.

In [3]:
RESUMES_URL = (
    "https://huggingface.co/datasets/"
    "michaelozon/candidate-matching-synthetic/"
    "resolve/main/resumes/train-00000-of-00001.parquet"
)

JOBS_URL = (
    "https://huggingface.co/datasets/"
    "michaelozon/candidate-matching-synthetic/"
    "resolve/main/jobs/train-00000-of-00001.parquet"
)

MATCHES_URL = (
    "https://huggingface.co/datasets/"
    "michaelozon/candidate-matching-synthetic/"
    "resolve/main/matches/train-00000-of-00001.parquet"
)

print("Dataset paths configured.")

Dataset paths configured.


In [4]:
resumes = load_dataset(
    "parquet",
    data_files=RESUMES_URL,
    split="train"
)

jobs = load_dataset(
    "parquet",
    data_files=JOBS_URL,
    split="train"
)

matches = load_dataset(
    "parquet",
    data_files=MATCHES_URL,
    split="train"
)

print("Resumes:", len(resumes))
print("Jobs:", len(jobs))
print("Matches:", len(matches))

Resumes: 10000
Jobs: 2500
Matches: 2500


## 3. Inspect the Dataset Structure

Before building a retrieval system, we inspect the fields available in each
dataset.

This prevents the retrieval experiments from depending on assumptions about
the schema.

In [5]:
print("Resume columns:")
print(resumes.column_names)

print("\nJob columns:")
print(jobs.column_names)

print("\nMatch columns:")
print(matches.column_names)

Resume columns:
['resume_id', 'role', 'seniority', 'years_experience', 'industry', 'education', 'skills', 'summary', 'experience_bullets']

Job columns:
['job_id', 'job_title', 'seniority', 'industry', 'must_have_skills', 'nice_to_have_skills', 'description', 'responsibilities', 'requirements']

Match columns:
['job_id', 'relevant_resume_ids']


In [6]:
print("Sample resume:")
print(resumes[0])

print("\nSample job:")
print(jobs[0])

print("\nSample relevance record:")
print(matches[0])

Sample resume:
{'resume_id': 'R_000000', 'role': 'Software Engineer', 'seniority': 'Senior', 'years_experience': 12, 'industry': 'EdTech', 'education': 'BSc', 'skills': ['OOP', 'Databases', 'Git', 'Docker', 'Python', 'Unit Testing', 'Java'], 'summary': 'Software Engineer with 12 years of experience in EdTech.', 'experience_bullets': ['Delivered results using structured workflows and clear communication', 'Collaborated with stakeholders to define needs and execute tasks', 'Maintained reporting and documentation to support team performance']}

Sample job:
{'job_id': 'J_000000', 'job_title': 'Project Manager', 'seniority': 'Senior', 'industry': 'Retail', 'must_have_skills': ['Timeline Management', 'Process Improvement', 'Stakeholder Communication'], 'nice_to_have_skills': ['Asana'], 'description': 'We are hiring a Project Manager to support teams in Retail.', 'responsibilities': ['Deliver high-quality work aligned with goals and timelines', 'Collaborate with stakeholders and communicate p

## 4. Construct Retrieval Text

The retrieval baseline requires a single textual representation for each job
and each resume.

For jobs, we use the job title, seniority, industry, explicit must-have and
nice-to-have skills, description, and requirements.

For resumes, we use the candidate's role, seniority, years of experience,
industry, education, skills, summary, and experience bullets.

The responsibility field is excluded from the initial baseline because the
dataset contains highly repetitive/template-like responsibility text. This
keeps the first retrieval experiment focused on information that differentiates
jobs and candidates.

The resulting representations are used only for retrieval experiments.

In [7]:
def list_to_text(value):
    if value is None:
        return ""
    
    if isinstance(value, (list, tuple)):
        return " ".join(str(item) for item in value)
    
    return str(value)


def build_job_text(job):
    parts = [
        f"Job title: {job['job_title']}",
        f"Seniority: {job['seniority']}",
        f"Industry: {job['industry']}",
        f"Must-have skills: {list_to_text(job['must_have_skills'])}",
        f"Nice-to-have skills: {list_to_text(job['nice_to_have_skills'])}",
        f"Description: {job['description']}",
        f"Requirements: {list_to_text(job['requirements'])}",
    ]
    
    return " ".join(parts)


def build_resume_text(resume):
    parts = [
        f"Role: {resume['role']}",
        f"Seniority: {resume['seniority']}",
        f"Years of experience: {resume['years_experience']}",
        f"Industry: {resume['industry']}",
        f"Education: {resume['education']}",
        f"Skills: {list_to_text(resume['skills'])}",
        f"Summary: {resume['summary']}",
        f"Experience: {list_to_text(resume['experience_bullets'])}",
    ]
    
    return " ".join(parts)


print("Text construction functions ready.")

Text construction functions ready.


In [8]:
sample_job_text = build_job_text(jobs[0])
sample_resume_text = build_resume_text(resumes[0])

print("JOB TEXT")
print(sample_job_text)

print("\n" + "=" * 80)

print("\nRESUME TEXT")
print(sample_resume_text)

JOB TEXT
Job title: Project Manager Seniority: Senior Industry: Retail Must-have skills: Timeline Management Process Improvement Stakeholder Communication Nice-to-have skills: Asana Description: We are hiring a Project Manager to support teams in Retail. Requirements: Relevant experience and ability to learn quickly Strong communication and ownership mindset Comfort working with tools, data, and cross-functional teams


RESUME TEXT
Role: Software Engineer Seniority: Senior Years of experience: 12 Industry: EdTech Education: BSc Skills: OOP Databases Git Docker Python Unit Testing Java Summary: Software Engineer with 12 years of experience in EdTech. Experience: Delivered results using structured workflows and clear communication Collaborated with stakeholders to define needs and execute tasks Maintained reporting and documentation to support team performance


## 5. Create Retrieval Corpus

We now convert the datasets into pandas DataFrames.

The resume corpus will be indexed once and reused across all retrieval
experiments.

For each job, the relevance mapping will provide the ground-truth set of
relevant resume IDs.

In [9]:
resumes_df = pd.DataFrame(resumes)
jobs_df = pd.DataFrame(jobs)
matches_df = pd.DataFrame(matches)

resumes_df["retrieval_text"] = resumes_df.apply(
    build_resume_text,
    axis=1
)

jobs_df["retrieval_text"] = jobs_df.apply(
    build_job_text,
    axis=1
)

print("Resume corpus:", resumes_df.shape)
print("Job corpus:", jobs_df.shape)
print("Matches:", matches_df.shape)

Resume corpus: (10000, 10)
Job corpus: (2500, 10)
Matches: (2500, 2)


In [10]:
matches_df["num_relevant"] = matches_df["relevant_resume_ids"].apply(len)

print(matches_df["num_relevant"].value_counts().sort_index())
print("\nAverage relevant resumes per job:",
      matches_df["num_relevant"].mean())

num_relevant
30    2500
Name: count, dtype: int64

Average relevant resumes per job: 30.0


## 6. TF-IDF Retrieval Baseline

TF-IDF represents each job and resume as a weighted bag-of-words vector.

The weight of a term increases when:

- the term is important within a document
- the term is relatively uncommon across the corpus

We then compute cosine similarity between the job vector and every resume
vector and rank resumes by similarity.

This is a lexical baseline: it primarily captures exact or closely related
word overlap and does not understand semantic similarity.

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("TF-IDF imports ready.")

TF-IDF imports ready.


In [12]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
)

resume_tfidf = tfidf_vectorizer.fit_transform(
    resumes_df["retrieval_text"]
)

print("Resume TF-IDF matrix shape:", resume_tfidf.shape)
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

Resume TF-IDF matrix shape: (10000, 1172)
Vocabulary size: 1172


### TF-IDF Querying

The resume corpus is transformed once into TF-IDF vectors.

For each incoming job, we transform the job into the same vector space and
compute cosine similarity against the entire resume corpus.

The result is a ranked list of candidate resumes.

In [13]:
def tfidf_retrieve(job_text, top_k=10):
    job_vector = tfidf_vectorizer.transform([job_text])
    
    scores = cosine_similarity(
        job_vector,
        resume_tfidf
    ).ravel()
    
    top_indices = np.argsort(-scores)[:top_k]
    
    results = resumes_df.iloc[top_indices][
        ["resume_id", "role", "seniority", "skills"]
    ].copy()
    
    results["score"] = scores[top_indices]
    
    return results.reset_index(drop=True)

In [14]:
test_job = jobs_df.iloc[0]

tfidf_results = tfidf_retrieve(
    test_job["retrieval_text"],
    top_k=10
)

print("Job:")
print(test_job["job_title"])

print("\nTop 10 retrieved resumes:")
tfidf_results

Job:
Project Manager

Top 10 retrieved resumes:


,resume_id,role,seniority,skills,score
0,R_003618,Project Manager,Junior,"[Timeline Management, Process Improvement, Pro...",0.619488
1,R_001048,Project Manager,Junior,"[Asana, Timeline Management, Stakeholder Commu...",0.617724
2,R_006175,Project Manager,Mid,"[Timeline Management, Asana, Reporting, KPIs, ...",0.615200
3,R_002794,Project Manager,Mid,"[Timeline Management, Asana, KPIs, Cross-funct...",0.607487
4,R_005404,Project Manager,Senior,"[Timeline Management, KPIs, Cross-functional C...",0.607051
5,R_009442,Project Manager,Mid,"[Timeline Management, Project Planning, Proces...",0.591883
6,R_007525,Project Manager,Junior,"[Stakeholder Communication, Cross-functional C...",0.587974
7,R_009701,Project Manager,Mid,"[Timeline Management, Risk Management, Jira, P...",0.583020
8,R_002345,Project Manager,Mid,"[Process Improvement, Stakeholder Communicatio...",0.580844
9,R_002963,Project Manager,Junior,"[Timeline Management, Cross-functional Coordin...",0.577929


In [15]:
test_job_id = test_job["job_id"]

ground_truth = set(
    matches_df.loc[
        matches_df["job_id"] == test_job_id,
        "relevant_resume_ids"
    ].iloc[0]
)

retrieved_ids = set(tfidf_results["resume_id"])

hits = retrieved_ids.intersection(ground_truth)

print("Job ID:", test_job_id)
print("Ground-truth relevant resumes:", len(ground_truth))
print("Retrieved resumes:", len(retrieved_ids))
print("Relevant resumes in top 10:", len(hits))
print("Recall@10 for this job:", len(hits) / len(ground_truth))

Job ID: J_000000
Ground-truth relevant resumes: 30
Retrieved resumes: 10
Relevant resumes in top 10: 0
Recall@10 for this job: 0.0


### Ground-Truth Inspection

Before evaluating the retrieval model across the full dataset, we inspect the
ground-truth candidates for the example job.

This helps verify that the relevance labels correspond to the information
contained in the job and resume records.

In [16]:
ground_truth_rows = resumes_df[
    resumes_df["resume_id"].isin(ground_truth)
][
    ["resume_id", "role", "seniority", "years_experience", "industry", "skills"]
].copy()

ground_truth_rows

,resume_id,role,seniority,years_experience,industry,skills
201,R_000201,Program Coordinator,Mid,6,FinTech,"[Reporting, Project Planning, Cross-functional..."
915,R_000915,Operations Manager,Mid,6,SaaS,"[Jira, Project Planning, Cross-functional Coor..."
1302,R_001302,Program Coordinator,Senior,8,Gaming,"[Asana, Risk Management, Process Improvement, ..."
2900,R_002900,Project Manager,Mid,6,Cybersecurity,"[Stakeholder Communication, Timeline Managemen..."
2906,R_002906,Operations Manager,Junior,1,SaaS,"[Stakeholder Communication, Cross-functional C..."
3326,R_003326,Operations Manager,Mid,2,EdTech,"[Risk Management, Stakeholder Communication, J..."
3468,R_003468,Operations Manager,Senior,9,Travel,"[Risk Management, Cross-functional Coordinatio..."
3626,R_003626,Program Coordinator,Senior,10,Travel,"[Stakeholder Communication, Project Planning, ..."
3641,R_003641,Program Coordinator,Senior,11,EdTech,"[KPIs, Cross-functional Coordination, Risk Man..."
3647,R_003647,Operations Manager,Mid,6,E-commerce,"[Asana, Timeline Management, Reporting, KPIs, ..."


### Retrieved vs Ground-Truth Candidates

The following comparison helps determine whether the benchmark's relevance
relationships align with lexical similarity and the explicit job attributes.

In [17]:
print("TOP 10 TF-IDF CANDIDATES")
display(tfidf_results)

print("\nGROUND-TRUTH CANDIDATES")
display(ground_truth_rows)

TOP 10 TF-IDF CANDIDATES


,resume_id,role,seniority,skills,score
0,R_003618,Project Manager,Junior,"[Timeline Management, Process Improvement, Pro...",0.619488
1,R_001048,Project Manager,Junior,"[Asana, Timeline Management, Stakeholder Commu...",0.617724
2,R_006175,Project Manager,Mid,"[Timeline Management, Asana, Reporting, KPIs, ...",0.615200
3,R_002794,Project Manager,Mid,"[Timeline Management, Asana, KPIs, Cross-funct...",0.607487
4,R_005404,Project Manager,Senior,"[Timeline Management, KPIs, Cross-functional C...",0.607051
5,R_009442,Project Manager,Mid,"[Timeline Management, Project Planning, Proces...",0.591883
6,R_007525,Project Manager,Junior,"[Stakeholder Communication, Cross-functional C...",0.587974
7,R_009701,Project Manager,Mid,"[Timeline Management, Risk Management, Jira, P...",0.583020
8,R_002345,Project Manager,Mid,"[Process Improvement, Stakeholder Communicatio...",0.580844
9,R_002963,Project Manager,Junior,"[Timeline Management, Cross-functional Coordin...",0.577929



GROUND-TRUTH CANDIDATES


,resume_id,role,seniority,years_experience,industry,skills
201,R_000201,Program Coordinator,Mid,6,FinTech,"[Reporting, Project Planning, Cross-functional..."
915,R_000915,Operations Manager,Mid,6,SaaS,"[Jira, Project Planning, Cross-functional Coor..."
1302,R_001302,Program Coordinator,Senior,8,Gaming,"[Asana, Risk Management, Process Improvement, ..."
2900,R_002900,Project Manager,Mid,6,Cybersecurity,"[Stakeholder Communication, Timeline Managemen..."
2906,R_002906,Operations Manager,Junior,1,SaaS,"[Stakeholder Communication, Cross-functional C..."
3326,R_003326,Operations Manager,Mid,2,EdTech,"[Risk Management, Stakeholder Communication, J..."
3468,R_003468,Operations Manager,Senior,9,Travel,"[Risk Management, Cross-functional Coordinatio..."
3626,R_003626,Program Coordinator,Senior,10,Travel,"[Stakeholder Communication, Project Planning, ..."
3641,R_003641,Program Coordinator,Senior,11,EdTech,"[KPIs, Cross-functional Coordination, Risk Man..."
3647,R_003647,Operations Manager,Mid,6,E-commerce,"[Asana, Timeline Management, Reporting, KPIs, ..."


In [18]:
job_must_have = set(
    jobs_df.loc[
        jobs_df["job_id"] == test_job_id,
        "must_have_skills"
    ].iloc[0]
)

print("Job must-have skills:")
print(job_must_have)

print("\nGround-truth candidates and skill overlap:")

for _, row in ground_truth_rows.iterrows():
    resume_skills = set(row["skills"])
    overlap = job_must_have.intersection(resume_skills)
    
    print(
        row["resume_id"],
        "|",
        row["role"],
        "| overlap:",
        overlap
    )

Job must-have skills:
{'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}

Ground-truth candidates and skill overlap:
R_000201 | Program Coordinator | overlap: {'Timeline Management', 'Process Improvement'}
R_000915 | Operations Manager | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_001302 | Program Coordinator | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_002900 | Project Manager | overlap: {'Timeline Management', 'Stakeholder Communication'}
R_002906 | Operations Manager | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_003326 | Operations Manager | overlap: {'Stakeholder Communication', 'Process Improvement'}
R_003468 | Operations Manager | overlap: {'Timeline Management', 'Process Improvement'}
R_003626 | Program Coordinator | overlap: {'Timeline Management', 'Stakeholder Communication'}
R_003641 | Program Coordinator | overlap: {'Timel

In [19]:
print("TF-IDF TOP 10 CANDIDATES AND SKILL OVERLAP")
print("=" * 80)

for _, row in tfidf_results.iterrows():
    resume_skills = set(row["skills"])
    overlap = job_must_have.intersection(resume_skills)

    print(
        row["resume_id"],
        "|",
        row["role"],
        "| score:",
        round(row["score"], 4),
        "| overlap:",
        overlap
    )

TF-IDF TOP 10 CANDIDATES AND SKILL OVERLAP
R_003618 | Project Manager | score: 0.6195 | overlap: {'Timeline Management', 'Process Improvement'}
R_001048 | Project Manager | score: 0.6177 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_006175 | Project Manager | score: 0.6152 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_002794 | Project Manager | score: 0.6075 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_005404 | Project Manager | score: 0.6071 | overlap: {'Timeline Management', 'Process Improvement'}
R_009442 | Project Manager | score: 0.5919 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_007525 | Project Manager | score: 0.588 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_009701 | Project Manager | score: 0.583 | overlap: {'Timeline Management', 'Process Improvement'}
R_0023

### TF-IDF Failure Analysis

The initial TF-IDF retrieval strongly favors candidates whose lexical
representation resembles the job title and overall job description.

However, the benchmark relevance labels are primarily aligned with explicit
skill compatibility. The example job requires Stakeholder Communication,
Timeline Management, and Process Improvement, and the ground-truth candidates
consistently contain these skills.

The first TF-IDF retrieval therefore provides an important baseline failure
case: lexical similarity alone does not reliably reproduce the benchmark's
skill-based relevance relationships.

This motivates evaluating retrieval methods that explicitly capture semantic
and skill-level similarity.

## 7. Retrieval Evaluation

A retrieval system should be evaluated across the complete benchmark rather
than on a single example.

For each job, the system produces a ranked list of candidate resumes.
The ranked list is compared against the known relevant resumes.

We use multiple ranking metrics:

- **Recall@K** — measures how many relevant candidates are retrieved within
  the top K results.
- **MRR (Mean Reciprocal Rank)** — measures how highly the first relevant
  candidate appears.
- **nDCG@K** — evaluates the quality of the ranking within the top K results.

Using multiple metrics is important because a retrieval system may retrieve
relevant candidates while still placing them too low in the ranking.

The same evaluation framework will be reused for TF-IDF, BM25, dense
retrieval, and hybrid retrieval.

In [20]:
from sklearn.metrics import ndcg_score

print("Evaluation imports ready.")

Evaluation imports ready.


### Recall@K

Recall@K measures the proportion of all relevant candidates that appear in
the top K retrieved results.

For this benchmark, each job has 30 known relevant resumes.

\[
Recall@K =
\frac{\text{relevant resumes retrieved in top K}}
{\text{total relevant resumes}}
\]

For example, if 3 of the 30 relevant candidates appear in the top 10:

\[
Recall@10 = \frac{3}{30} = 0.10
\]

In [21]:
def recall_at_k(retrieved_ids, relevant_ids, k):
    retrieved_top_k = retrieved_ids[:k]
    relevant_set = set(relevant_ids)

    hits = sum(
        1
        for resume_id in retrieved_top_k
        if resume_id in relevant_set
    )

    return hits / len(relevant_set)


def reciprocal_rank(retrieved_ids, relevant_ids):
    relevant_set = set(relevant_ids)

    for rank, resume_id in enumerate(retrieved_ids, start=1):
        if resume_id in relevant_set:
            return 1.0 / rank

    return 0.0


print("Evaluation functions ready.")

Evaluation functions ready.


### nDCG@K

nDCG evaluates the ordering of relevant candidates within the top K results.

A relevant candidate appearing near the top contributes more than the same
candidate appearing much lower in the ranking.

This is useful for candidate retrieval because the system should not only
retrieve relevant candidates but should prioritize them near the top of the
candidate list.

In [22]:
def ndcg_at_k(retrieved_ids, relevant_ids, k):
    relevant_set = set(relevant_ids)

    top_k_ids = retrieved_ids[:k]

    relevance = np.array([
        1 if resume_id in relevant_set else 0
        for resume_id in top_k_ids
    ])

    if len(relevance) == 0 or relevance.sum() == 0:
        return 0.0

    # Higher score means a higher-ranked position.
    ranking_scores = np.arange(
        len(relevance),
        0,
        -1
    )

    return ndcg_score(
        [relevance],
        [ranking_scores]
    )


print("Corrected nDCG function ready.")

Corrected nDCG function ready.


### Metric Sanity Check

Before running the complete benchmark, we test the metric functions on the
TF-IDF ranking for the example job.

This verifies that the evaluation implementation behaves as expected before
scaling it to all jobs.

In [23]:
test_retrieved_ids = tfidf_results["resume_id"].tolist()
test_relevant_ids = list(ground_truth)

print("Recall@1 :", recall_at_k(test_retrieved_ids, test_relevant_ids, 1))
print("Recall@3 :", recall_at_k(test_retrieved_ids, test_relevant_ids, 3))
print("Recall@5 :", recall_at_k(test_retrieved_ids, test_relevant_ids, 5))
print("Recall@10:", recall_at_k(test_retrieved_ids, test_relevant_ids, 10))

print("MRR      :", reciprocal_rank(
    test_retrieved_ids,
    test_relevant_ids
))

print("nDCG@5   :", ndcg_at_k(
    test_retrieved_ids,
    test_relevant_ids,
    5
))

print("nDCG@10  :", ndcg_at_k(
    test_retrieved_ids,
    test_relevant_ids,
    10
))

Recall@1 : 0.0
Recall@3 : 0.0
Recall@5 : 0.0
Recall@10: 0.0
MRR      : 0.0
nDCG@5   : 0.0
nDCG@10  : 0.0


## 8. Full TF-IDF Benchmark

The evaluation framework is now validated on the example job.

We next evaluate TF-IDF across all jobs in the benchmark.

For each job:

1. Retrieve the candidate ranking using TF-IDF.
2. Compare the ranking against the ground-truth relevant candidates.
3. Compute Recall@1, Recall@3, Recall@5, Recall@10, MRR, nDCG@5,
   and nDCG@10.
4. Aggregate the results across all jobs.

This produces the first complete retrieval baseline for SkillAlign.

In [24]:
def evaluate_tfidf():
    results = []

    for _, job in jobs_df.iterrows():

        job_id = job["job_id"]

        relevant_ids = matches_df.loc[
            matches_df["job_id"] == job_id,
            "relevant_resume_ids"
        ].iloc[0]

        job_vector = tfidf_vectorizer.transform(
            [job["retrieval_text"]]
        )

        scores = cosine_similarity(
            job_vector,
            resume_tfidf
        ).ravel()

        ranked_indices = np.argsort(-scores)

        ranked_resume_ids = resumes_df.iloc[
            ranked_indices
        ]["resume_id"].tolist()

        results.append({
            "job_id": job_id,
            "recall@1": recall_at_k(
                ranked_resume_ids,
                relevant_ids,
                1
            ),
            "recall@3": recall_at_k(
                ranked_resume_ids,
                relevant_ids,
                3
            ),
            "recall@5": recall_at_k(
                ranked_resume_ids,
                relevant_ids,
                5
            ),
            "recall@10": recall_at_k(
                ranked_resume_ids,
                relevant_ids,
                10
            ),
            "mrr": reciprocal_rank(
                ranked_resume_ids,
                relevant_ids
            ),
            "ndcg@5": ndcg_at_k(
                ranked_resume_ids,
                relevant_ids,
                5
            ),
            "ndcg@10": ndcg_at_k(
                ranked_resume_ids,
                relevant_ids,
                10
            ),
        })

    return pd.DataFrame(results)


tfidf_eval = evaluate_tfidf()

print("Evaluation complete.")
print("Number of jobs evaluated:", len(tfidf_eval))

tfidf_eval.head()

Evaluation complete.
Number of jobs evaluated: 2500


,job_id,recall@1,recall@3,recall@5,recall@10,mrr,ndcg@5,ndcg@10
0,J_000000,0.0,0.0,0.0,0.0,0.025641,0.0,0.0
1,J_000001,0.0,0.0,0.0,0.0,0.083333,0.0,0.0
2,J_000002,0.0,0.0,0.0,0.0,0.020000,0.0,0.0
3,J_000003,0.0,0.0,0.0,0.0,0.066667,0.0,0.0
4,J_000004,0.0,0.0,0.0,0.0,0.019608,0.0,0.0


### TF-IDF Aggregate Results

The per-job metrics are aggregated across the complete benchmark.

The mean is reported for each retrieval metric so that the performance of the
retrieval system can be compared consistently with later baselines.

In [25]:
tfidf_summary = pd.DataFrame({
    "Metric": [
        "Recall@1",
        "Recall@3",
        "Recall@5",
        "Recall@10",
        "MRR",
        "nDCG@5",
        "nDCG@10",
    ],
    "TF-IDF": [
        tfidf_eval["recall@1"].mean(),
        tfidf_eval["recall@3"].mean(),
        tfidf_eval["recall@5"].mean(),
        tfidf_eval["recall@10"].mean(),
        tfidf_eval["mrr"].mean(),
        tfidf_eval["ndcg@5"].mean(),
        tfidf_eval["ndcg@10"].mean(),
    ]
})

tfidf_summary

,Metric,TF-IDF
0,Recall@1,0.001267
1,Recall@3,0.003240
2,Recall@5,0.005640
3,Recall@10,0.011173
4,MRR,0.119255
5,nDCG@5,0.095746
6,nDCG@10,0.139092


## TF-IDF Baseline Results

TF-IDF was evaluated across all 2,500 jobs using the 10,000-resume candidate
corpus.

The baseline achieved:

- Recall@1: 0.001267
- Recall@3: 0.003240
- Recall@5: 0.005640
- Recall@10: 0.011173
- MRR: 0.119255
- nDCG@5: 0.095746
- nDCG@10: 0.139092

The low Recall@10 indicates that lexical TF-IDF retrieval does not closely
reproduce the benchmark's candidate relevance relationships.

However, the non-zero MRR and nDCG values indicate that the lexical
representation still provides some ranking signal.

This establishes TF-IDF as the first reproducible retrieval baseline for
comparison with BM25 and semantic retrieval methods.

## 9. BM25 Retrieval Baseline

BM25 is a probabilistic lexical retrieval method commonly used for
information retrieval.

Unlike TF-IDF, BM25 explicitly accounts for:

- term frequency saturation
- inverse document frequency
- document length normalization

BM25 provides a stronger lexical retrieval baseline before moving to dense
semantic representations.

The same evaluation metrics used for TF-IDF will be applied so that the
retrieval methods can be compared directly.

In [26]:
import importlib.util

if importlib.util.find_spec("rank_bm25") is None:
    print("rank_bm25 is not installed.")
else:
    print("rank_bm25 is available.")

rank_bm25 is available.


In [27]:
from rank_bm25 import BM25Okapi

print("BM25 import ready.")

BM25 import ready.


### BM25 Corpus Construction

BM25 operates on tokenized documents rather than TF-IDF vectors.

We tokenize the resume corpus once and construct a BM25 index. The same
tokenization procedure is then applied to each job query.

In [28]:
def tokenize(text):
    return text.lower().split()


bm25_corpus = [
    tokenize(text)
    for text in resumes_df["retrieval_text"]
]

bm25 = BM25Okapi(bm25_corpus)

print("BM25 corpus size:", len(bm25_corpus))
print("BM25 index ready.")

BM25 corpus size: 10000
BM25 index ready.


### BM25 Retrieval

For each job, BM25 assigns a relevance score to every resume in the corpus.

The resumes are then ranked by decreasing BM25 score.

The retrieval interface is kept consistent with TF-IDF so that both methods
can be evaluated using the same benchmark metrics.

In [29]:
def bm25_retrieve(job_text, top_k=10):
    query_tokens = tokenize(job_text)

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(-scores)[:top_k]

    results = resumes_df.iloc[top_indices][
        ["resume_id", "role", "seniority", "skills"]
    ].copy()

    results["score"] = scores[top_indices]

    return results.reset_index(drop=True)

### BM25 Sanity Check

Before evaluating BM25 across the full benchmark, we inspect its ranking for
the same example job used for the TF-IDF baseline.

Using the same job makes the comparison between lexical retrieval methods
directly interpretable.

In [30]:
bm25_results = bm25_retrieve(
    test_job["retrieval_text"],
    top_k=10
)

print("Job:")
print(test_job["job_title"])

print("\nTop 10 BM25 retrieved resumes:")
bm25_results

Job:
Project Manager

Top 10 BM25 retrieved resumes:


,resume_id,role,seniority,skills,score
0,R_005978,Project Manager,Mid,"[Stakeholder Communication, Process Improvemen...",29.271096
1,R_008109,Project Manager,Junior,"[Jira, Cross-functional Coordination, Risk Man...",29.271096
2,R_008322,Project Manager,Junior,"[Project Planning, Cross-functional Coordinati...",29.085571
3,R_005404,Project Manager,Senior,"[Timeline Management, KPIs, Cross-functional C...",28.508051
4,R_001048,Project Manager,Junior,"[Asana, Timeline Management, Stakeholder Commu...",28.371609
5,R_002794,Project Manager,Mid,"[Timeline Management, Asana, KPIs, Cross-funct...",28.371609
6,R_000062,Project Manager,Senior,"[KPIs, Jira, Project Planning, Process Improve...",28.327994
7,R_009665,Project Manager,Junior,"[Project Planning, Jira, KPIs, Process Improve...",27.613763
8,R_003561,Project Manager,Senior,"[Timeline Management, Jira, Reporting, Process...",27.586818
9,R_003618,Project Manager,Junior,"[Timeline Management, Process Improvement, Pro...",27.441323


In [31]:
print("BM25 TOP 10 CANDIDATES AND SKILL OVERLAP")
print("=" * 80)

for _, row in bm25_results.iterrows():
    resume_skills = set(row["skills"])
    overlap = job_must_have.intersection(resume_skills)

    print(
        row["resume_id"],
        "|",
        row["role"],
        "| score:",
        round(row["score"], 4),
        "| overlap:",
        overlap
    )

BM25 TOP 10 CANDIDATES AND SKILL OVERLAP
R_005978 | Project Manager | score: 29.2711 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_008109 | Project Manager | score: 29.2711 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_008322 | Project Manager | score: 29.0856 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_005404 | Project Manager | score: 28.5081 | overlap: {'Timeline Management', 'Process Improvement'}
R_001048 | Project Manager | score: 28.3716 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_002794 | Project Manager | score: 28.3716 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_000062 | Project Manager | score: 28.328 | overlap: {'Timeline Management', 'Process Improvement'}
R_009665 | Project Manager | score: 27.6138 | overlap: {'Timeline Management', 'Process Improvement'}

## 10. Full BM25 Benchmark

The BM25 retrieval system is now evaluated across the complete benchmark.

The same evaluation protocol used for TF-IDF is retained:

- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR
- nDCG@5
- nDCG@10

Using the same metrics allows the lexical retrieval methods to be compared
directly.

In [32]:
def evaluate_bm25():
    results = []

    for _, job in jobs_df.iterrows():

        job_id = job["job_id"]

        relevant_ids = matches_df.loc[
            matches_df["job_id"] == job_id,
            "relevant_resume_ids"
        ].iloc[0]

        query_tokens = tokenize(job["retrieval_text"])

        scores = bm25.get_scores(query_tokens)

        ranked_indices = np.argsort(-scores)

        ranked_resume_ids = resumes_df.iloc[
            ranked_indices
        ]["resume_id"].tolist()

        results.append({
            "job_id": job_id,
            "recall@1": recall_at_k(
                ranked_resume_ids,
                relevant_ids,
                1
            ),
            "recall@3": recall_at_k(
                ranked_resume_ids,
                relevant_ids,
                3
            ),
            "recall@5": recall_at_k(
                ranked_resume_ids,
                relevant_ids,
                5
            ),
            "recall@10": recall_at_k(
                ranked_resume_ids,
                relevant_ids,
                10
            ),
            "mrr": reciprocal_rank(
                ranked_resume_ids,
                relevant_ids
            ),
            "ndcg@5": ndcg_at_k(
                ranked_resume_ids,
                relevant_ids,
                5
            ),
            "ndcg@10": ndcg_at_k(
                ranked_resume_ids,
                relevant_ids,
                10
            ),
        })

    return pd.DataFrame(results)


bm25_eval = evaluate_bm25()

print("BM25 evaluation complete.")
print("Number of jobs evaluated:", len(bm25_eval))

bm25_eval.head()

BM25 evaluation complete.
Number of jobs evaluated: 2500


,job_id,recall@1,recall@3,recall@5,recall@10,mrr,ndcg@5,ndcg@10
0,J_000000,0.0,0.0,0.0,0.0,0.010753,0.0,0.0
1,J_000001,0.0,0.0,0.0,0.0,0.017241,0.0,0.0
2,J_000002,0.0,0.0,0.0,0.0,0.026316,0.0,0.0
3,J_000003,0.0,0.0,0.0,0.0,0.022727,0.0,0.0
4,J_000004,0.0,0.0,0.0,0.0,0.012658,0.0,0.0


### BM25 Aggregate Results

The per-job BM25 metrics are aggregated across the complete benchmark.

The resulting averages will be compared with the TF-IDF baseline.

In [33]:
bm25_summary = pd.DataFrame({
    "Metric": [
        "Recall@1",
        "Recall@3",
        "Recall@5",
        "Recall@10",
        "MRR",
        "nDCG@5",
        "nDCG@10",
    ],
    "BM25": [
        bm25_eval["recall@1"].mean(),
        bm25_eval["recall@3"].mean(),
        bm25_eval["recall@5"].mean(),
        bm25_eval["recall@10"].mean(),
        bm25_eval["mrr"].mean(),
        bm25_eval["ndcg@5"].mean(),
        bm25_eval["ndcg@10"].mean(),
    ]
})

bm25_summary

,Metric,BM25
0,Recall@1,0.001333
1,Recall@3,0.003720
2,Recall@5,0.006173
3,Recall@10,0.011747
4,MRR,0.126812
5,nDCG@5,0.104730
6,nDCG@10,0.147283


## 11. TF-IDF vs BM25

The two lexical retrieval methods are compared using identical evaluation
metrics.

This comparison establishes whether BM25 provides a stronger lexical
retrieval baseline than TF-IDF before introducing dense semantic retrieval.

In [34]:
lexical_comparison = tfidf_summary.merge(
    bm25_summary,
    on="Metric"
)

lexical_comparison

,Metric,TF-IDF,BM25
0,Recall@1,0.001267,0.001333
1,Recall@3,0.003240,0.003720
2,Recall@5,0.005640,0.006173
3,Recall@10,0.011173,0.011747
4,MRR,0.119255,0.126812
5,nDCG@5,0.095746,0.104730
6,nDCG@10,0.139092,0.147283


## BM25 Baseline Results

BM25 was evaluated across all 2,500 jobs using the same 10,000-resume corpus
and evaluation protocol as TF-IDF.

BM25 achieved:

- Recall@1: 0.001333
- Recall@3: 0.003720
- Recall@5: 0.006173
- Recall@10: 0.011747
- MRR: 0.126812
- nDCG@5: 0.104730
- nDCG@10: 0.147283

BM25 outperformed TF-IDF across all reported metrics, although the
improvements were modest.

This establishes BM25 as the stronger of the two lexical baselines while
also showing that lexical retrieval alone has limited agreement with the
benchmark relevance relationships.

The next stage therefore evaluates dense semantic retrieval.

## Benchmark Relevance Analysis

Before introducing dense semantic retrieval, we inspect how the benchmark's
relevance labels relate to explicit job and resume attributes.

The synthetic dataset provides job-level relevance relationships but does not
provide requirement-level explanations for why a resume is considered
relevant.

Therefore, retrieval metrics should be interpreted as agreement with the
dataset's relevance construction rather than as a complete measure of
real-world candidate suitability.

This diagnostic helps distinguish retrieval-model limitations from possible
properties of the synthetic benchmark.

In [35]:
# Compare skill-overlap distributions for benchmark-relevant
# and randomly sampled non-relevant candidates.

rng = np.random.default_rng(42)

overlap_records = []

for _, job in jobs_df.sample(
    n=100,
    random_state=42
).iterrows():

    job_id = job["job_id"]

    relevant_ids = set(
        matches_df.loc[
            matches_df["job_id"] == job_id,
            "relevant_resume_ids"
        ].iloc[0]
    )

    job_skills = set(job["must_have_skills"])

    relevant_sample = resumes_df[
        resumes_df["resume_id"].isin(relevant_ids)
    ]

    non_relevant_pool = resumes_df[
        ~resumes_df["resume_id"].isin(relevant_ids)
    ]

    sample_size = min(
        len(relevant_sample),
        len(non_relevant_pool)
    )

    non_relevant_sample = non_relevant_pool.sample(
        n=sample_size,
        random_state=42
    )

    for _, resume in relevant_sample.iterrows():
        overlap = len(
            job_skills.intersection(set(resume["skills"]))
        )

        overlap_records.append({
            "label": "relevant",
            "skill_overlap": overlap
        })

    for _, resume in non_relevant_sample.iterrows():
        overlap = len(
            job_skills.intersection(set(resume["skills"]))
        )

        overlap_records.append({
            "label": "non_relevant",
            "skill_overlap": overlap
        })


overlap_df = pd.DataFrame(overlap_records)

overlap_df.groupby("label")["skill_overlap"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
non_relevant,3000.0,0.425333,0.989656,0.0,0.0,0.0,0.0,5.0
relevant,3000.0,3.078667,0.743413,2.0,3.0,3.0,3.0,5.0


## 12. Skill-Aware Retrieval Baseline

The previous TF-IDF and BM25 systems represented the complete job and resume
as a single text field.

The benchmark analysis showed that explicit skill overlap is strongly
associated with relevance. We therefore introduce a skill-aware lexical
baseline.

Instead of treating every field equally, this baseline gives greater
importance to explicit skill information.

For the initial experiment, BM25 is applied to a representation containing:

- job title
- seniority
- industry
- must-have skills
- nice-to-have skills

and resume:

- role
- seniority
- industry
- skills

This tests whether retrieval performance improves when the representation is
more closely aligned with the benchmark's relevance signal.

In [36]:
def build_skill_aware_job_text(job):
    parts = [
        f"Job title: {job['job_title']}",
        f"Seniority: {job['seniority']}",
        f"Industry: {job['industry']}",
        f"Must-have skills: {list_to_text(job['must_have_skills'])}",
        f"Nice-to-have skills: {list_to_text(job['nice_to_have_skills'])}",
    ]

    return " ".join(parts)


def build_skill_aware_resume_text(resume):
    parts = [
        f"Role: {resume['role']}",
        f"Seniority: {resume['seniority']}",
        f"Industry: {resume['industry']}",
        f"Skills: {list_to_text(resume['skills'])}",
    ]

    return " ".join(parts)


jobs_df["skill_aware_text"] = jobs_df.apply(
    build_skill_aware_job_text,
    axis=1
)

resumes_df["skill_aware_text"] = resumes_df.apply(
    build_skill_aware_resume_text,
    axis=1
)

print("Skill-aware representations created.")

Skill-aware representations created.


### Skill-Aware BM25 Index

A separate BM25 index is constructed using the skill-aware resume
representation.

The original BM25 index remains unchanged so that the two retrieval systems
can be compared directly.

In [37]:
skill_aware_corpus = [
    tokenize(text)
    for text in resumes_df["skill_aware_text"]
]

skill_aware_bm25 = BM25Okapi(skill_aware_corpus)

print("Skill-aware BM25 corpus size:", len(skill_aware_corpus))
print("Skill-aware BM25 index ready.")

Skill-aware BM25 corpus size: 10000
Skill-aware BM25 index ready.


### Skill-Aware BM25 Retrieval

The skill-aware BM25 system uses the same ranking mechanism as the original
BM25 baseline, but operates on the reduced representation.

This isolates the effect of the text representation from the retrieval
algorithm itself.

In [38]:
def skill_aware_bm25_retrieve(job_text, top_k=10):
    query_tokens = tokenize(job_text)

    scores = skill_aware_bm25.get_scores(query_tokens)

    top_indices = np.argsort(-scores)[:top_k]

    results = resumes_df.iloc[top_indices][
        ["resume_id", "role", "seniority", "skills"]
    ].copy()

    results["score"] = scores[top_indices]

    return results.reset_index(drop=True)

### Skill-Aware Retrieval Sanity Check

The same example job used for TF-IDF and BM25 is evaluated using the
skill-aware representation.

This allows us to determine whether explicitly emphasizing skill-related
fields changes the retrieved candidates.

In [39]:
test_job = jobs_df.iloc[0]

skill_aware_results = skill_aware_bm25_retrieve(
    test_job["skill_aware_text"],
    top_k=10
)

print("Job:")
print(test_job["job_title"])

print("\nTop 10 skill-aware BM25 candidates:")
skill_aware_results

Job:
Project Manager

Top 10 skill-aware BM25 candidates:


,resume_id,role,seniority,skills,score
0,R_006597,Operations Manager,Senior,"[Process Improvement, Asana, Stakeholder Commu...",21.592000
1,R_002312,Project Manager,Junior,"[Project Planning, Jira, Stakeholder Communica...",20.836576
2,R_002924,Project Manager,Junior,"[Process Improvement, Timeline Management, Rep...",20.726581
3,R_008322,Project Manager,Junior,"[Project Planning, Cross-functional Coordinati...",20.365103
4,R_001048,Project Manager,Junior,"[Asana, Timeline Management, Stakeholder Commu...",20.359353
5,R_002794,Project Manager,Mid,"[Timeline Management, Asana, KPIs, Cross-funct...",20.359353
6,R_005978,Project Manager,Mid,"[Stakeholder Communication, Process Improvemen...",20.278355
7,R_008109,Project Manager,Junior,"[Jira, Cross-functional Coordination, Risk Man...",20.278355
8,R_003706,Project Manager,Senior,"[Process Improvement, Jira, Stakeholder Commun...",20.277602
9,R_003183,Program Coordinator,Senior,"[Project Planning, Risk Management, Asana, Sta...",20.106540


In [40]:
print("SKILL-AWARE BM25 TOP 10 AND SKILL OVERLAP")
print("=" * 80)

for _, row in skill_aware_results.iterrows():
    resume_skills = set(row["skills"])
    overlap = job_must_have.intersection(resume_skills)

    print(
        row["resume_id"],
        "|",
        row["role"],
        "| score:",
        round(row["score"], 4),
        "| overlap:",
        overlap
    )

SKILL-AWARE BM25 TOP 10 AND SKILL OVERLAP
R_006597 | Operations Manager | score: 21.592 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_002312 | Project Manager | score: 20.8366 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_002924 | Project Manager | score: 20.7266 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_008322 | Project Manager | score: 20.3651 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_001048 | Project Manager | score: 20.3594 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_002794 | Project Manager | score: 20.3594 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_005978 | Project Manager | score: 20.2784 | overlap: {'Timeline Management', 'Stakeholder Communication', 'Process Improvement'}
R_008109 | Project Manager | score: 20.

## Skill-Aware BM25 Evaluation

The skill-aware representation is evaluated across the complete benchmark
using the same retrieval metrics as TF-IDF and standard BM25.

This experiment tests whether explicitly emphasizing job and resume skill
fields improves retrieval performance.

In [41]:
def evaluate_skill_aware_bm25():
    results = []

    for _, job in jobs_df.iterrows():

        job_id = job["job_id"]

        relevant_ids = matches_df.loc[
            matches_df["job_id"] == job_id,
            "relevant_resume_ids"
        ].iloc[0]

        query_tokens = tokenize(job["skill_aware_text"])

        scores = skill_aware_bm25.get_scores(query_tokens)

        ranked_indices = np.argsort(-scores)

        ranked_resume_ids = resumes_df.iloc[
            ranked_indices
        ]["resume_id"].tolist()

        results.append({
            "job_id": job_id,
            "recall@1": recall_at_k(ranked_resume_ids, relevant_ids, 1),
            "recall@3": recall_at_k(ranked_resume_ids, relevant_ids, 3),
            "recall@5": recall_at_k(ranked_resume_ids, relevant_ids, 5),
            "recall@10": recall_at_k(ranked_resume_ids, relevant_ids, 10),
            "mrr": reciprocal_rank(ranked_resume_ids, relevant_ids),
            "ndcg@5": ndcg_at_k(ranked_resume_ids, relevant_ids, 5),
            "ndcg@10": ndcg_at_k(ranked_resume_ids, relevant_ids, 10),
        })

    return pd.DataFrame(results)


skill_aware_eval = evaluate_skill_aware_bm25()

skill_aware_summary = pd.DataFrame({
    "Metric": [
        "Recall@1",
        "Recall@3",
        "Recall@5",
        "Recall@10",
        "MRR",
        "nDCG@5",
        "nDCG@10",
    ],
    "TF-IDF": [
        tfidf_eval["recall@1"].mean(),
        tfidf_eval["recall@3"].mean(),
        tfidf_eval["recall@5"].mean(),
        tfidf_eval["recall@10"].mean(),
        tfidf_eval["mrr"].mean(),
        tfidf_eval["ndcg@5"].mean(),
        tfidf_eval["ndcg@10"].mean(),
    ],
    "BM25": [
        bm25_eval["recall@1"].mean(),
        bm25_eval["recall@3"].mean(),
        bm25_eval["recall@5"].mean(),
        bm25_eval["recall@10"].mean(),
        bm25_eval["mrr"].mean(),
        bm25_eval["ndcg@5"].mean(),
        bm25_eval["ndcg@10"].mean(),
    ],
    "Skill-Aware BM25": [
        skill_aware_eval["recall@1"].mean(),
        skill_aware_eval["recall@3"].mean(),
        skill_aware_eval["recall@5"].mean(),
        skill_aware_eval["recall@10"].mean(),
        skill_aware_eval["mrr"].mean(),
        skill_aware_eval["ndcg@5"].mean(),
        skill_aware_eval["ndcg@10"].mean(),
    ],
})

print("Skill-aware BM25 evaluation complete.")
print("Jobs evaluated:", len(skill_aware_eval))

skill_aware_summary

Skill-aware BM25 evaluation complete.
Jobs evaluated: 2500


,Metric,TF-IDF,BM25,Skill-Aware BM25
0,Recall@1,0.001267,0.001333,0.001173
1,Recall@3,0.003240,0.003720,0.003773
2,Recall@5,0.005640,0.006173,0.006013
3,Recall@10,0.011173,0.011747,0.012067
4,MRR,0.119255,0.126812,0.124371
5,nDCG@5,0.095746,0.104730,0.099527
6,nDCG@10,0.139092,0.147283,0.147293


## Lexical Retrieval Conclusions

Three lexical retrieval strategies were evaluated across all 2,500 jobs:

- TF-IDF
- BM25
- Skill-Aware BM25

BM25 provided the strongest and most consistent lexical baseline. Skill-aware
BM25 produced small improvements on some recall metrics but did not improve
the overall ranking quality consistently.

Therefore, standard BM25 is retained as the lexical retrieval baseline.

The results motivate the next experiment: dense semantic retrieval using
retrieval-oriented embedding models.

## BGE-M3 Dense Retrieval

BGE-M3 is evaluated as the first semantic retrieval model.
Unlike TF-IDF and BM25, dense retrieval represents the job and resume
as contextual embeddings, allowing semantically related terms to match
even when the exact wording differs.

The same retrieval benchmark and metrics are used for a direct comparison
with the lexical baselines.

In [42]:
import torch
from FlagEmbedding import BGEM3FlagModel

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

bge_model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=True
)

print("BGE-M3 loaded.")

PyTorch: 2.14.0+cu130
CUDA: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE-M3 loaded.


In [43]:
import time
test_texts = resumes_df["retrieval_text"].iloc[:100].tolist()

start = time.perf_counter()

test_output = bge_model.encode(
    test_texts,
    batch_size=2,
    max_length=512,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)

elapsed = time.perf_counter() - start

print(f"100 resumes: {elapsed:.2f}s")
print(f"Per resume: {elapsed / 100:.3f}s")
print(f"Estimated 10,000 resumes: {elapsed * 100 / 60:.2f} min")
print("Embedding shape:", test_output["dense_vecs"].shape)

Inference Embeddings: 100%|██████████| 50/50 [00:02<00:00, 17.07it/s]

100 resumes: 7.47s
Per resume: 0.075s
Estimated 10,000 resumes: 12.46 min
Embedding shape: (100, 1024)


In [44]:
# Benchmark a larger batch size before full-corpus encoding

start = time.perf_counter()

test_output_b4 = bge_model.encode(
    test_texts,
    batch_size=4,
    max_length=512,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)

elapsed_b4 = time.perf_counter() - start

print(f"Batch size 4 - 100 resumes: {elapsed_b4:.2f}s")
print(f"Per resume: {elapsed_b4 / 100:.3f}s")
print(f"Estimated 10,000 resumes: {elapsed_b4 * 100 / 60:.2f} min")
print("Embedding shape:", test_output_b4["dense_vecs"].shape)

Inference Embeddings: 100%|██████████| 25/25 [00:01<00:00, 18.29it/s]

Batch size 4 - 100 resumes: 1.52s
Per resume: 0.015s
Estimated 10,000 resumes: 2.53 min
Embedding shape: (100, 1024)


In [45]:
# Full BGE-M3 dense encoding

BGE_BATCH_SIZE = 4
BGE_MAX_LENGTH = 512

start = time.perf_counter()

resume_output = bge_model.encode(
    resumes_df["retrieval_text"].tolist(),
    batch_size=BGE_BATCH_SIZE,
    max_length=BGE_MAX_LENGTH,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)

resume_embeddings = resume_output["dense_vecs"]

print(f"Resume encoding complete: {resume_embeddings.shape}")

job_output = bge_model.encode(
    jobs_df["retrieval_text"].tolist(),
    batch_size=BGE_BATCH_SIZE,
    max_length=BGE_MAX_LENGTH,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)

job_embeddings = job_output["dense_vecs"]

encoding_time = time.perf_counter() - start

print(f"Job encoding complete: {job_embeddings.shape}")
print(f"Total encoding time: {encoding_time / 60:.2f} minutes")

np.save("data/bge_m3_resume_embeddings.npy", resume_embeddings)
np.save("data/bge_m3_job_embeddings.npy", job_embeddings)

print("Embeddings saved.")

Inference Embeddings: 100%|██████████| 2500/2500 [01:58<00:00, 21.09it/s]


Resume encoding complete: (10000, 1024)


Inference Embeddings: 100%|██████████| 625/625 [00:30<00:00, 20.64it/s]

Job encoding complete: (2500, 1024)
Total encoding time: 2.54 minutes
Embeddings saved.


In [46]:
from pathlib import Path

data_dir = Path.cwd() / "data"
data_dir.mkdir(parents=True, exist_ok=True)

np.save(data_dir / "bge_m3_resume_embeddings.npy", resume_embeddings)
np.save(data_dir / "bge_m3_job_embeddings.npy", job_embeddings)

print("Embeddings saved to:")
print(data_dir / "bge_m3_resume_embeddings.npy")
print(data_dir / "bge_m3_job_embeddings.npy")

Embeddings saved to:
C:\Users\USER\OneDrive\Desktop\TANISHA_PRAKASH\SkillAlign\notebooks\data\bge_m3_resume_embeddings.npy
C:\Users\USER\OneDrive\Desktop\TANISHA_PRAKASH\SkillAlign\notebooks\data\bge_m3_job_embeddings.npy


In [47]:
print(
    "Resume file size:",
    round((data_dir / "bge_m3_resume_embeddings.npy").stat().st_size / 1024**2, 2),
    "MB"
)

print(
    "Job file size:",
    round((data_dir / "bge_m3_job_embeddings.npy").stat().st_size / 1024**2, 2),
    "MB"
)

Resume file size: 19.53 MB
Job file size: 4.88 MB


In [48]:
def evaluate_bge_dense():
    results = []

    for job_idx, job in jobs_df.iterrows():
        job_id = job["job_id"]

        relevant_ids = matches_df.loc[
            matches_df["job_id"] == job_id,
            "relevant_resume_ids"
        ].iloc[0]

        scores = resume_embeddings @ job_embeddings[job_idx]

        ranked_indices = np.argsort(-scores)
        ranked_resume_ids = resumes_df.iloc[
            ranked_indices
        ]["resume_id"].tolist()

        results.append({
            "job_id": job_id,
            "recall@1": recall_at_k(ranked_resume_ids, relevant_ids, 1),
            "recall@3": recall_at_k(ranked_resume_ids, relevant_ids, 3),
            "recall@5": recall_at_k(ranked_resume_ids, relevant_ids, 5),
            "recall@10": recall_at_k(ranked_resume_ids, relevant_ids, 10),
            "mrr": reciprocal_rank(ranked_resume_ids, relevant_ids),
            "ndcg@5": ndcg_at_k(ranked_resume_ids, relevant_ids, 5),
            "ndcg@10": ndcg_at_k(ranked_resume_ids, relevant_ids, 10),
        })

    return pd.DataFrame(results)

In [49]:
# BGE-M3 Dense Retrieval Evaluation

bge_dense_eval = evaluate_bge_dense()

bge_dense_results = bge_dense_eval.mean(numeric_only=True)

print(bge_dense_results)

recall@1     0.001080
recall@3     0.003253
recall@5     0.005240
recall@10    0.010133
mrr          0.111658
ndcg@5       0.089916
ndcg@10      0.126693
dtype: float64


## BGE-M3 Dense Retrieval

BGE-M3 dense retrieval achieved:

- Recall@1: 0.001080
- Recall@3: 0.003253
- Recall@5: 0.005240
- Recall@10: 0.010133
- MRR: 0.111658
- nDCG@5: 0.089916
- nDCG@10: 0.126693

On this synthetic benchmark, BGE-M3 dense retrieval did not outperform the
lexical baselines. BM25 remained stronger across the primary ranking metrics.

This result suggests that the benchmark's relevance labels are strongly
aligned with explicit lexical/skill overlap. BGE-M3 dense retrieval is
therefore retained as a semantic baseline rather than replacing BM25.

The next experiment evaluates BGE-M3 sparse and hybrid retrieval to determine
whether combining lexical and semantic signals improves ranking quality.

In [50]:
# BGE-M3 sparse retrieval

sparse_output = bge_model.encode(
    resumes_df["retrieval_text"].tolist(),
    batch_size=4,
    max_length=512,
    return_dense=False,
    return_sparse=True
)

job_sparse_output = bge_model.encode(
    jobs_df["retrieval_text"].tolist(),
    batch_size=4,
    max_length=512,
    return_dense=False,
    return_sparse=True
)

resume_sparse = sparse_output["lexical_weights"]
job_sparse = job_sparse_output["lexical_weights"]

print("Resume sparse vectors:", len(resume_sparse))
print("Job sparse vectors:", len(job_sparse))

Inference Embeddings: 100%|██████████| 625/625 [00:30<00:00, 20.75it/s]

Resume sparse vectors: 10000
Job sparse vectors: 2500


In [53]:
from scipy.sparse import csr_matrix

def sparse_dicts_to_matrix(sparse_dicts):
    rows = []
    cols = []
    data = []

    for row_idx, weights in enumerate(sparse_dicts):
        for token_id, weight in weights.items():
            rows.append(row_idx)
            cols.append(token_id)
            data.append(float(weight))  # float16 → Python float

    return csr_matrix(
        (np.asarray(data, dtype=np.float32), (rows, cols)),
        shape=(len(sparse_dicts), bge_model.tokenizer.vocab_size),
        dtype=np.float32
    )

resume_sparse_matrix = sparse_dicts_to_matrix(resume_sparse)
job_sparse_matrix = sparse_dicts_to_matrix(job_sparse)

print("Resume sparse matrix:", resume_sparse_matrix.shape)
print("Job sparse matrix:", job_sparse_matrix.shape)

Resume sparse matrix: (10000, 250002)
Job sparse matrix: (2500, 250002)


In [52]:
bge_sparse_eval = evaluate_bge_sparse_fast()
bge_sparse_results = bge_sparse_eval.mean(numeric_only=True)

print(bge_sparse_results)

NameError: name 'evaluate_bge_sparse_fast' is not defined

In [ ]:
def evaluate_bge_sparse_fast():
    results = []
    chunk_size = 100

    for start in range(0, len(jobs_df), chunk_size):
        end = min(start + chunk_size, len(jobs_df))

        # Vectorized sparse dot product
        scores_matrix = (
            job_sparse_matrix[start:end] @ resume_sparse_matrix.T
        ).toarray()

        for local_idx, scores in enumerate(scores_matrix):
            job_idx = start + local_idx
            job_id = jobs_df.iloc[job_idx]["job_id"]

            relevant_ids = matches_df.loc[
                matches_df["job_id"] == job_id,
                "relevant_resume_ids"
            ].iloc[0]

            ranked_indices = np.argsort(-scores)

            ranked_resume_ids = resumes_df.iloc[
                ranked_indices
            ]["resume_id"].tolist()

            results.append({
                "job_id": job_id,
                "recall@1": recall_at_k(
                    ranked_resume_ids, relevant_ids, 1
                ),
                "recall@3": recall_at_k(
                    ranked_resume_ids, relevant_ids, 3
                ),
                "recall@5": recall_at_k(
                    ranked_resume_ids, relevant_ids, 5
                ),
                "recall@10": recall_at_k(
                    ranked_resume_ids, relevant_ids, 10
                ),
                "mrr": reciprocal_rank(
                    ranked_resume_ids, relevant_ids
                ),
                "ndcg@5": ndcg_at_k(
                    ranked_resume_ids, relevant_ids, 5
                ),
                "ndcg@10": ndcg_at_k(
                    ranked_resume_ids, relevant_ids, 10
                ),
            })

        if end % 500 == 0 or end == len(jobs_df):
            print(f"Evaluated {end}/{len(jobs_df)} jobs")

    return pd.DataFrame(results)

In [ ]:
bge_sparse_eval = evaluate_bge_sparse_fast()
bge_sparse_results = bge_sparse_eval.mean(numeric_only=True)

print(bge_sparse_results)

In [ ]:
def minmax_normalize(scores):
    min_score = scores.min()
    max_score = scores.max()

    if max_score == min_score:
        return np.zeros_like(scores, dtype=np.float32)

    return (scores - min_score) / (max_score - min_score)


def evaluate_bge_hybrid(alpha):
    """
    alpha = weight assigned to dense retrieval.
    (1 - alpha) = weight assigned to sparse retrieval.
    """
    results = []
    chunk_size = 100

    for start in range(0, len(jobs_df), chunk_size):
        end = min(start + chunk_size, len(jobs_df))

        # Dense scores
        dense_scores_matrix = (
            job_embeddings[start:end] @ resume_embeddings.T
        )

        # Sparse scores
        sparse_scores_matrix = (
            job_sparse_matrix[start:end] @ resume_sparse_matrix.T
        ).toarray()

        for local_idx in range(end - start):
            job_idx = start + local_idx
            job_id = jobs_df.iloc[job_idx]["job_id"]

            relevant_ids = matches_df.loc[
                matches_df["job_id"] == job_id,
                "relevant_resume_ids"
            ].iloc[0]

            dense_scores = dense_scores_matrix[local_idx]
            sparse_scores = sparse_scores_matrix[local_idx]

            # Normalize each signal independently
            dense_norm = minmax_normalize(dense_scores)
            sparse_norm = minmax_normalize(sparse_scores)

            # Weighted hybrid score
            hybrid_scores = (
                alpha * dense_norm
                + (1 - alpha) * sparse_norm
            )

            ranked_indices = np.argsort(-hybrid_scores)

            ranked_resume_ids = resumes_df.iloc[
                ranked_indices
            ]["resume_id"].tolist()

            results.append({
                "job_id": job_id,
                "recall@1": recall_at_k(
                    ranked_resume_ids, relevant_ids, 1
                ),
                "recall@3": recall_at_k(
                    ranked_resume_ids, relevant_ids, 3
                ),
                "recall@5": recall_at_k(
                    ranked_resume_ids, relevant_ids, 5
                ),
                "recall@10": recall_at_k(
                    ranked_resume_ids, relevant_ids, 10
                ),
                "mrr": reciprocal_rank(
                    ranked_resume_ids, relevant_ids
                ),
                "ndcg@5": ndcg_at_k(
                    ranked_resume_ids, relevant_ids, 5
                ),
                "ndcg@10": ndcg_at_k(
                    ranked_resume_ids, relevant_ids, 10
                ),
            })

        if end % 500 == 0 or end == len(jobs_df):
            print(f"Evaluated {end}/{len(jobs_df)} jobs")

    return pd.DataFrame(results)

In [ ]:
alphas = [0.0, 0.25, 0.50, 0.75, 1.0]

hybrid_results = {}

for alpha in alphas:
    print(f"\nDense weight = {alpha:.2f}")

    evaluation = evaluate_bge_hybrid(alpha)

    hybrid_results[alpha] = evaluation.mean(
        numeric_only=True
    )

hybrid_comparison = pd.DataFrame(hybrid_results).T

hybrid_comparison.index.name = "dense_weight"

print(hybrid_comparison)

In [ ]:
from sentence_transformers import SentenceTransformer

qwen_model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cuda"
)

print("Device:", qwen_model.device)

In [ ]:
import pandas as pd
import numpy as np

resumes_df = pd.read_parquet(
    "https://huggingface.co/datasets/michaelozon/candidate-matching-synthetic/resolve/main/resumes/train-00000-of-00001.parquet"
)

jobs_df = pd.read_parquet(
    "https://huggingface.co/datasets/michaelozon/candidate-matching-synthetic/resolve/main/jobs/train-00000-of-00001.parquet"
)

matches_df = pd.read_parquet(
    "https://huggingface.co/datasets/michaelozon/candidate-matching-synthetic/resolve/main/matches/train-00000-of-00001.parquet"
)

print("Resumes:", len(resumes_df))
print("Jobs:", len(jobs_df))
print("Matches:", len(matches_df))

In [ ]:
def list_to_text(value):
    if value is None:
        return ""

    if isinstance(value, (list, tuple)):
        return " ".join(str(item) for item in value)

    return str(value)


def build_job_text(job):
    return " ".join([
        f"Job title: {job['job_title']}",
        f"Seniority: {job['seniority']}",
        f"Industry: {job['industry']}",
        f"Must-have skills: {list_to_text(job['must_have_skills'])}",
        f"Nice-to-have skills: {list_to_text(job['nice_to_have_skills'])}",
        f"Description: {job['description']}",
        f"Requirements: {list_to_text(job['requirements'])}",
    ])


def build_resume_text(resume):
    return " ".join([
        f"Role: {resume['role']}",
        f"Seniority: {resume['seniority']}",
        f"Years of experience: {resume['years_experience']}",
        f"Industry: {resume['industry']}",
        f"Education: {resume['education']}",
        f"Skills: {list_to_text(resume['skills'])}",
        f"Summary: {resume['summary']}",
        f"Experience: {list_to_text(resume['experience_bullets'])}",
    ])


resumes_df["retrieval_text"] = resumes_df.apply(
    build_resume_text, axis=1
)

jobs_df["retrieval_text"] = jobs_df.apply(
    build_job_text, axis=1
)

In [ ]:

import time

sample_texts = resumes_df["retrieval_text"].iloc[:100].tolist()

start = time.time()

sample_embeddings = qwen_model.encode(
    sample_texts,
    batch_size=4,
    show_progress_bar=True,
    normalize_embeddings=True
)

elapsed = time.time() - start

print("Shape:", sample_embeddings.shape)
print(f"Time: {elapsed:.2f}s")
print(f"Estimated 10k resumes: {elapsed * 100 / 60:.2f} min")

In [ ]:
import time

start = time.time()

sample_embeddings_b8 = qwen_model.encode(
    sample_texts,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True
)

elapsed_b8 = time.time() - start

print("Shape:", sample_embeddings_b8.shape)
print(f"Time: {elapsed_b8:.2f}s")
print(f"Estimated 10k resumes: {elapsed_b8 * 100 / 60:.2f} min")

In [ ]:
import time
import numpy as np

# Qwen3 dense embeddings
start = time.time()

qwen_resume_embeddings = qwen_model.encode(
    resumes_df["retrieval_text"].tolist(),
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True
)

resume_time = time.time() - start

print(f"Resume embeddings: {qwen_resume_embeddings.shape}")
print(f"Resume encoding time: {resume_time / 60:.2f} min")

start = time.time()

qwen_job_embeddings = qwen_model.encode(
    jobs_df["retrieval_text"].tolist(),
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True
)

job_time = time.time() - start

print(f"Job embeddings: {qwen_job_embeddings.shape}")
print(f"Job encoding time: {job_time / 60:.2f} min")

print(f"Total encoding time: {(resume_time + job_time) / 60:.2f} min")

In [ ]:
np.save("data/qwen3_embedding_resume_embeddings.npy", qwen_resume_embeddings)
np.save("data/qwen3_embedding_job_embeddings.npy", qwen_job_embeddings)

print("Saved Qwen3 embeddings.")

In [ ]:
import numpy as np

def recall_at_k(ranked_ids, relevant_ids, k):
    return len(set(ranked_ids[:k]) & set(relevant_ids)) / len(relevant_ids)

def reciprocal_rank(ranked_ids, relevant_ids):
    relevant_ids = set(relevant_ids)
    for rank, rid in enumerate(ranked_ids, start=1):
        if rid in relevant_ids:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(ranked_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)

    dcg = 0.0
    for i, rid in enumerate(ranked_ids[:k]):
        if rid in relevant_ids:
            dcg += 1.0 / np.log2(i + 2)

    ideal_hits = min(len(relevant_ids), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))

    return dcg / idcg if idcg > 0 else 0.0


resume_ids = resumes_df["resume_id"].tolist()
resume_id_to_idx = {rid: i for i, rid in enumerate(resume_ids)}

results = {
    "recall@1": [],
    "recall@3": [],
    "recall@5": [],
    "recall@10": [],
    "mrr": [],
    "ndcg@5": [],
    "ndcg@10": [],
}

# Evaluate jobs in chunks to avoid creating a huge score matrix at once
job_chunk_size = 100

for start in range(0, len(jobs_df), job_chunk_size):
    end = min(start + job_chunk_size, len(jobs_df))

    scores = qwen_job_embeddings[start:end] @ qwen_resume_embeddings.T

    for local_i, row_idx in enumerate(range(start, end)):
        job_id = jobs_df.iloc[row_idx]["job_id"]

        relevant_row = matches_df[matches_df["job_id"] == job_id].iloc[0]
        relevant_ids = relevant_row["relevant_resume_ids"]

        top_indices = np.argsort(-scores[local_i])[:10]
        ranked_ids = [resume_ids[i] for i in top_indices]

        results["recall@1"].append(
            recall_at_k(ranked_ids, relevant_ids, 1)
        )
        results["recall@3"].append(
            recall_at_k(ranked_ids, relevant_ids, 3)
        )
        results["recall@5"].append(
            recall_at_k(ranked_ids, relevant_ids, 5)
        )
        results["recall@10"].append(
            recall_at_k(ranked_ids, relevant_ids, 10)
        )
        results["mrr"].append(
            reciprocal_rank(ranked_ids, relevant_ids)
        )
        results["ndcg@5"].append(
            ndcg_at_k(ranked_ids, relevant_ids, 5)
        )
        results["ndcg@10"].append(
            ndcg_at_k(ranked_ids, relevant_ids, 10)
        )

qwen_metrics = {metric: np.mean(values) for metric, values in results.items()}

print("Qwen3-Embedding-0.6B Retrieval Results")
print("-" * 45)

for metric, value in qwen_metrics.items():
    print(f"{metric.upper():<12}: {value:.6f}")

In [ ]:
import time
import numpy as np

start = time.time()

qwen_job_embeddings_query = qwen_model.encode(
    jobs_df["retrieval_text"].tolist(),
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True,
    prompt_name="query"
)

elapsed = time.time() - start

print("Job embeddings:", qwen_job_embeddings_query.shape)
print(f"Encoding time: {elapsed / 60:.2f} min")

np.save(
    "data/qwen3_embedding_job_embeddings_query.npy",
    qwen_job_embeddings_query
)

print("Saved query-prompt job embeddings.")

In [ ]:
# Evaluate Qwen3 with query-prompted job embeddings

results = {
    "recall@1": [],
    "recall@3": [],
    "recall@5": [],
    "recall@10": [],
    "mrr": [],
    "ndcg@5": [],
    "ndcg@10": [],
}

for start in range(0, len(jobs_df), 100):
    end = min(start + 100, len(jobs_df))

    scores = (
        qwen_job_embeddings_query[start:end]
        @ qwen_resume_embeddings.T
    )

    for local_i, row_idx in enumerate(range(start, end)):
        job_id = jobs_df.iloc[row_idx]["job_id"]

        relevant_row = matches_df[
            matches_df["job_id"] == job_id
        ].iloc[0]

        relevant_ids = relevant_row["relevant_resume_ids"]

        top_indices = np.argsort(-scores[local_i])[:10]
        ranked_ids = [resume_ids[i] for i in top_indices]

        results["recall@1"].append(
            recall_at_k(ranked_ids, relevant_ids, 1)
        )
        results["recall@3"].append(
            recall_at_k(ranked_ids, relevant_ids, 3)
        )
        results["recall@5"].append(
            recall_at_k(ranked_ids, relevant_ids, 5)
        )
        results["recall@10"].append(
            recall_at_k(ranked_ids, relevant_ids, 10)
        )
        results["mrr"].append(
            reciprocal_rank(ranked_ids, relevant_ids)
        )
        results["ndcg@5"].append(
            ndcg_at_k(ranked_ids, relevant_ids, 5)
        )
        results["ndcg@10"].append(
            ndcg_at_k(ranked_ids, relevant_ids, 10)
        )

qwen_query_metrics = {
    metric: np.mean(values)
    for metric, values in results.items()
}

print("Qwen3-Embedding-0.6B — Query Prompt")
print("-" * 45)

for metric, value in qwen_query_metrics.items():
    print(f"{metric.upper():<12}: {value:.6f}")

In [ ]:
retrieval_ks = [10, 25, 50, 100]

for k in retrieval_ks:
    recall_scores = []

    for start in range(0, len(jobs_df), 100):
        end = min(start + 100, len(jobs_df))

        scores = (
            qwen_job_embeddings_query[start:end]
            @ qwen_resume_embeddings.T
        )

        for local_i, row_idx in enumerate(range(start, end)):
            job_id = jobs_df.iloc[row_idx]["job_id"]

            relevant_row = matches_df[
                matches_df["job_id"] == job_id
            ].iloc[0]

            relevant_ids = set(relevant_row["relevant_resume_ids"])

            top_indices = np.argsort(-scores[local_i])[:k]
            ranked_ids = [resume_ids[i] for i in top_indices]

            hits = len(set(ranked_ids) & relevant_ids)
            recall_scores.append(hits / len(relevant_ids))

    print(f"Recall@{k}: {np.mean(recall_scores):.6f}")

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().split()

bm25_corpus = [
    tokenize(text)
    for text in resumes_df["retrieval_text"]
]

bm25 = BM25Okapi(bm25_corpus)

resume_ids = resumes_df["resume_id"].tolist()

print("BM25 rebuilt.")
print(f"Corpus size: {len(bm25_corpus)}")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import csr_matrix
import numpy as np

# Same whitespace tokenization used by the earlier BM25 baseline
vectorizer = CountVectorizer(
    lowercase=True,
    tokenizer=lambda x: x.lower().split(),
    preprocessor=None,
    token_pattern=None
)

X = vectorizer.fit_transform(resumes_df["retrieval_text"])

N = X.shape[0]
doc_lengths = np.asarray(X.sum(axis=1)).ravel()
avgdl = doc_lengths.mean()

k1 = 1.5
b = 0.75
epsilon = 0.25

# Document frequency
df = np.asarray((X > 0).sum(axis=0)).ravel()

# rank_bm25-style IDF
idf = np.log((N - df + 0.5) / (df + 0.5))

average_idf = idf.mean()
idf[idf < 0] = epsilon * average_idf

# Row index for every non-zero entry
row_ids = np.repeat(np.arange(N), np.diff(X.indptr))

# BM25 denominator normalization
norm = k1 * (1 - b + b * doc_lengths[row_ids] / avgdl)

data = X.data.astype(np.float32)

bm25_values = (
    idf[X.indices]
    * (data * (k1 + 1))
    / (data + norm)
)

bm25_matrix = csr_matrix(
    (bm25_values, X.indices, X.indptr),
    shape=X.shape,
    dtype=np.float32
)

print("BM25 matrix:", bm25_matrix.shape)
print("Vocabulary:", len(vectorizer.vocabulary_))

In [ ]:
retrieval_ks = [10, 25, 50, 100]
recall_scores = {k: [] for k in retrieval_ks}

job_vectorizer = vectorizer

for start in range(0, len(jobs_df), 100):
    end = min(start + 100, len(jobs_df))

    query_matrix = job_vectorizer.transform(
        jobs_df["retrieval_text"].iloc[start:end]
    )

    # (10000 resumes × 100 jobs) sparse matrix multiplication
    scores = bm25_matrix @ query_matrix.T

    for local_i, row_idx in enumerate(range(start, end)):
        job_id = jobs_df.iloc[row_idx]["job_id"]

        relevant_row = matches_df[
            matches_df["job_id"] == job_id
        ].iloc[0]

        relevant_ids = set(relevant_row["relevant_resume_ids"])

        # Top 100 only
        job_scores = scores[:, local_i].toarray().ravel()
        top_indices = np.argpartition(-job_scores, 100)[:100]
        top_indices = top_indices[
            np.argsort(-job_scores[top_indices])
        ]

        ranked_ids = [resume_ids[i] for i in top_indices]

        for k in retrieval_ks:
            hits = len(set(ranked_ids[:k]) & relevant_ids)
            recall_scores[k].append(hits / len(relevant_ids))

    if end % 500 == 0 or end == len(jobs_df):
        print(f"Processed {end}/{len(jobs_df)} jobs")

print("\nBM25 Candidate-Pool Recall")
for k in retrieval_ks:
    print(f"Recall@{k}: {np.mean(recall_scores[k]):.6f}")

In [ ]:
def build_skill_aware_job_text(job):
    return " ".join([
        f"Job title: {job['job_title']}",
        f"Seniority: {job['seniority']}",
        f"Industry: {job['industry']}",
        f"Must-have skills: {list_to_text(job['must_have_skills'])}",
        f"Nice-to-have skills: {list_to_text(job['nice_to_have_skills'])}",
    ])


def build_skill_aware_resume_text(resume):
    return " ".join([
        f"Role: {resume['role']}",
        f"Seniority: {resume['seniority']}",
        f"Industry: {resume['industry']}",
        f"Skills: {list_to_text(resume['skills'])}",
    ])


resumes_df["skill_aware_text"] = resumes_df.apply(
    build_skill_aware_resume_text,
    axis=1
)

jobs_df["skill_aware_text"] = jobs_df.apply(
    build_skill_aware_job_text,
    axis=1
)

print("Skill-aware text rebuilt.")
print("Resume examples:", resumes_df["skill_aware_text"].head(2).tolist())

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import csr_matrix
import numpy as np

skill_vectorizer = CountVectorizer(
    lowercase=True,
    tokenizer=lambda x: x.lower().split(),
    preprocessor=None,
    token_pattern=None
)

X_skill = skill_vectorizer.fit_transform(
    resumes_df["skill_aware_text"]
)

N = X_skill.shape[0]
doc_lengths = np.asarray(X_skill.sum(axis=1)).ravel()
avgdl = doc_lengths.mean()

k1 = 1.5
b = 0.75
epsilon = 0.25

df = np.asarray((X_skill > 0).sum(axis=0)).ravel()

idf = np.log((N - df + 0.5) / (df + 0.5))

average_idf = idf.mean()
idf[idf < 0] = epsilon * average_idf

row_ids = np.repeat(
    np.arange(N),
    np.diff(X_skill.indptr)
)

norm = k1 * (
    1 - b + b * doc_lengths[row_ids] / avgdl
)

data = X_skill.data.astype(np.float32)

bm25_skill_values = (
    idf[X_skill.indices]
    * (data * (k1 + 1))
    / (data + norm)
)

bm25_skill_matrix = csr_matrix(
    (
        bm25_skill_values,
        X_skill.indices,
        X_skill.indptr
    ),
    shape=X_skill.shape,
    dtype=np.float32
)

print("Skill-aware BM25 matrix:", bm25_skill_matrix.shape)
print("Vocabulary:", len(skill_vectorizer.vocabulary_))

In [ ]:
retrieval_ks = [10, 25, 50, 100]
skill_recall_scores = {k: [] for k in retrieval_ks}

for start in range(0, len(jobs_df), 100):
    end = min(start + 100, len(jobs_df))

    query_matrix = skill_vectorizer.transform(
        jobs_df["skill_aware_text"].iloc[start:end]
    )

    scores = bm25_skill_matrix @ query_matrix.T

    for local_i, row_idx in enumerate(range(start, end)):
        job_id = jobs_df.iloc[row_idx]["job_id"]

        relevant_row = matches_df[
            matches_df["job_id"] == job_id
        ].iloc[0]

        relevant_ids = set(
            relevant_row["relevant_resume_ids"]
        )

        job_scores = scores[:, local_i].toarray().ravel()

        top_indices = np.argpartition(
            -job_scores, 100
        )[:100]

        top_indices = top_indices[
            np.argsort(-job_scores[top_indices])
        ]

        ranked_ids = [
            resume_ids[i] for i in top_indices
        ]

        for k in retrieval_ks:
            hits = len(
                set(ranked_ids[:k]) & relevant_ids
            )
            skill_recall_scores[k].append(
                hits / len(relevant_ids)
            )

print("\nSkill-Aware BM25 Candidate-Pool Recall")

for k in retrieval_ks:
    print(
        f"Recall@{k}: "
        f"{np.mean(skill_recall_scores[k]):.6f}"
    )

In [ ]:
hybrid_recall = []

for start in range(0, len(jobs_df), 100):
    end = min(start + 100, len(jobs_df))

    query_matrix = skill_vectorizer.transform(
        jobs_df["skill_aware_text"].iloc[start:end]
    )

    bm25_scores = bm25_skill_matrix @ query_matrix.T

    qwen_scores = (
        qwen_job_embeddings_query[start:end]
        @ qwen_resume_embeddings.T
    )

    for local_i, row_idx in enumerate(range(start, end)):
        job_id = jobs_df.iloc[row_idx]["job_id"]

        relevant_row = matches_df[
            matches_df["job_id"] == job_id
        ].iloc[0]

        relevant_ids = set(
            relevant_row["relevant_resume_ids"]
        )

        # Top 50 from each retriever
        bm25_scores_job = (
            bm25_scores[:, local_i].toarray().ravel()
        )

        bm25_top50 = np.argpartition(
            -bm25_scores_job, 50
        )[:50]

        qwen_top50 = np.argpartition(
            -qwen_scores[local_i], 50
        )[:50]

        # Union + deduplication
        hybrid_indices = set(bm25_top50) | set(qwen_top50)

        hybrid_ids = {
            resume_ids[i]
            for i in hybrid_indices
        }

        hits = len(hybrid_ids & relevant_ids)

        hybrid_recall.append(
            hits / len(relevant_ids)
        )

print(
    f"Hybrid Top-50 + Top-50 Recall@100: "
    f"{np.mean(hybrid_recall):.6f}"
)

In [ ]:
import torch
import transformers
import sentence_transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

JINA_MODEL = "jinaai/jina-embeddings-v3-hf"

jina_tokenizer = AutoTokenizer.from_pretrained(JINA_MODEL)

jina_model = AutoModel.from_pretrained(
    JINA_MODEL,
    dtype=torch.float16
)

jina_model = jina_model.to("cuda")
jina_model.eval()

print("Jina model loaded.")
print("Device:", next(jina_model.parameters()).device)

In [ ]:
del jina_model
torch.cuda.empty_cache()

In [ ]:
from transformers import AutoModel, AutoTokenizer

JINA_MODEL = "jinaai/jina-embeddings-v3-hf"

jina_tokenizer = AutoTokenizer.from_pretrained(JINA_MODEL)

jina_passage_model = AutoModel.from_pretrained(
    JINA_MODEL,
    dtype=torch.float16
)

jina_passage_model.load_adapter(
    JINA_MODEL,
    adapter_name="retrieval_passage",
    adapter_kwargs={"subfolder": "retrieval_passage"}
)

jina_passage_model.set_adapter("retrieval_passage")
jina_passage_model = jina_passage_model.to("cuda")
jina_passage_model.eval()

print("Clean Jina passage model loaded.")
print("Device:", next(jina_passage_model.parameters()).device)

In [ ]:
import time
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

JINA_MODEL = "jinaai/jina-embeddings-v3-hf"

jina_tokenizer = AutoTokenizer.from_pretrained(JINA_MODEL)

jina_query_model = AutoModel.from_pretrained(
    JINA_MODEL,
    dtype=torch.float16
)

jina_query_model.load_adapter(
    JINA_MODEL,
    adapter_name="retrieval_query",
    adapter_kwargs={"subfolder": "retrieval_query"}
)

jina_query_model.set_adapter("retrieval_query")
jina_query_model = jina_query_model.to("cuda")
jina_query_model.eval()

print("Jina query model loaded.")
print("Device:", next(jina_query_model.parameters()).device)

In [ ]:
sample_jobs = jobs_df["retrieval_text"].iloc[:100].tolist()

def encode_jina_model(model, texts, batch_size=4, max_length=512):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        encoded = jina_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoded = {
            key: value.to("cuda")
            for key, value in encoded.items()
        }

        with torch.no_grad():
            outputs = model(**encoded)

        mask = encoded["attention_mask"].unsqueeze(-1)

        embeddings = (
            outputs.last_hidden_state * mask
        ).sum(dim=1) / mask.sum(dim=1)

        embeddings = F.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(
            embeddings.cpu().numpy()
        )

    return np.vstack(all_embeddings)


start = time.time()

jina_query_sample = encode_jina_model(
    jina_query_model,
    sample_jobs,
    batch_size=4,
    max_length=512
)

elapsed = time.time() - start

print("Shape:", jina_query_sample.shape)
print(f"Time: {elapsed:.2f}s")
print(f"Texts/sec: {100 / elapsed:.2f}")

In [ ]:
del jina_query_model
torch.cuda.empty_cache()

print("Query model released.")

In [ ]:
import time
import numpy as np
import torch
import torch.nn.functional as F

sample_resumes = resumes_df["retrieval_text"].iloc[:100].tolist()

start = time.time()

all_embeddings = []

for start_idx in range(0, len(sample_resumes), 4):
    batch = sample_resumes[start_idx:start_idx + 4]

    encoded = jina_tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    encoded = {
        key: value.to("cuda")
        for key, value in encoded.items()
    }

    with torch.no_grad():
        outputs = jina_passage_model(**encoded)

    mask = encoded["attention_mask"].unsqueeze(-1)

    embeddings = (
        outputs.last_hidden_state * mask
    ).sum(dim=1) / mask.sum(dim=1)

    embeddings = F.normalize(
        embeddings,
        p=2,
        dim=1
    )

    all_embeddings.append(
        embeddings.cpu().numpy()
    )

jina_passage_sample = np.vstack(all_embeddings)

elapsed = time.time() - start

print("Shape:", jina_passage_sample.shape)
print(f"Time: {elapsed:.2f}s")
print(f"Texts/sec: {100 / elapsed:.2f}")
print(
    f"Estimated 10,000 resumes: "
    f"{elapsed * 100 / 60:.2f} min"
)

In [ ]:
import time
import numpy as np
import torch
import torch.nn.functional as F

start = time.time()

all_embeddings_b8 = []

for start_idx in range(0, len(sample_resumes), 8):
    batch = sample_resumes[start_idx:start_idx + 8]

    encoded = jina_tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    encoded = {
        key: value.to("cuda")
        for key, value in encoded.items()
    }

    with torch.no_grad():
        outputs = jina_passage_model(**encoded)

    mask = encoded["attention_mask"].unsqueeze(-1)

    embeddings = (
        outputs.last_hidden_state * mask
    ).sum(dim=1) / mask.sum(dim=1)

    embeddings = F.normalize(
        embeddings,
        p=2,
        dim=1
    )

    all_embeddings_b8.append(
        embeddings.cpu().numpy()
    )

jina_passage_sample_b8 = np.vstack(all_embeddings_b8)

elapsed_b8 = time.time() - start

print("Shape:", jina_passage_sample_b8.shape)
print(f"Time: {elapsed_b8:.2f}s")
print(f"Texts/sec: {100 / elapsed_b8:.2f}")
print(
    f"Estimated 10,000 resumes: "
    f"{elapsed_b8 * 100 / 60:.2f} min"
)

In [ ]:
del jina_passage_model
torch.cuda.empty_cache()

print("Passage model released.")

In [ ]:
from transformers import AutoModel

jina_query_model = AutoModel.from_pretrained(
    JINA_MODEL,
    dtype=torch.float16
)

jina_query_model.load_adapter(
    JINA_MODEL,
    adapter_name="retrieval_query",
    adapter_kwargs={"subfolder": "retrieval_query"}
)

jina_query_model.set_adapter("retrieval_query")
jina_query_model = jina_query_model.to("cuda")
jina_query_model.eval()

print("Clean Jina query model loaded.")
print("Device:", next(jina_query_model.parameters()).device)

In [ ]:
start = time.time()

jina_query_sample_b8 = encode_jina_model(
    jina_query_model,
    sample_jobs,
    batch_size=8,
    max_length=512
)

elapsed_b8 = time.time() - start

print("Shape:", jina_query_sample_b8.shape)
print(f"Time: {elapsed_b8:.2f}s")
print(f"Texts/sec: {100 / elapsed_b8:.2f}")
print(
    f"Estimated 2,500 jobs: "
    f"{elapsed_b8 * 25 / 60:.2f} min"
)

In [ ]:
import time

start = time.time()

jina_job_embeddings = encode_jina_model(
    jina_query_model,
    jobs_df["retrieval_text"].tolist(),
    batch_size=8,
    max_length=512
)

elapsed = time.time() - start

print("Job embeddings:", jina_job_embeddings.shape)
print(f"Encoding time: {elapsed / 60:.2f} min")

np.save(
    "data/jina_v3_job_embeddings.npy",
    jina_job_embeddings
)

print("Saved Jina job embeddings.")

In [ ]:
del jina_query_model
torch.cuda.empty_cache()

print("Query model released.")

In [ ]:
from transformers import AutoModel

jina_passage_model = AutoModel.from_pretrained(
    JINA_MODEL,
    dtype=torch.float16
)

jina_passage_model.load_adapter(
    JINA_MODEL,
    adapter_name="retrieval_passage",
    adapter_kwargs={"subfolder": "retrieval_passage"}
)

jina_passage_model.set_adapter("retrieval_passage")
jina_passage_model = jina_passage_model.to("cuda")
jina_passage_model.eval()

print("Clean Jina passage model loaded.")
print("Device:", next(jina_passage_model.parameters()).device)

In [ ]:
import time

start = time.time()

jina_resume_embeddings = encode_jina_model(
    jina_passage_model,
    resumes_df["retrieval_text"].tolist(),
    batch_size=8,
    max_length=512
)

elapsed = time.time() - start

print("Resume embeddings:", jina_resume_embeddings.shape)
print(f"Encoding time: {elapsed / 60:.2f} min")

np.save(
    "data/jina_v3_resume_embeddings.npy",
    jina_resume_embeddings
)

print("Saved Jina resume embeddings.")

In [ ]:
import numpy as np

# Recover saved embeddings if the variables are not currently in memory
if "jina_job_embeddings" not in globals():
    jina_job_embeddings = np.load(
        "data/jina_v3_job_embeddings.npy"
    )

if "jina_resume_embeddings" not in globals():
    jina_resume_embeddings = np.load(
        "data/jina_v3_resume_embeddings.npy"
    )

resume_ids = resumes_df["resume_id"].tolist()

# Map job_id -> relevant resume IDs once
relevant_map = dict(
    zip(
        matches_df["job_id"],
        matches_df["relevant_resume_ids"]
    )
)

metrics = {
    "recall@1": [],
    "recall@3": [],
    "recall@5": [],
    "recall@10": [],
    "recall@25": [],
    "recall@50": [],
    "recall@100": [],
    "mrr": [],
    "ndcg@5": [],
    "ndcg@10": [],
}

ks = [1, 3, 5, 10, 25, 50, 100]

for start in range(0, len(jobs_df), 100):
    end = min(start + 100, len(jobs_df))

    # Jina embeddings are already L2-normalized,
    # so dot product = cosine similarity
    scores = (
        jina_job_embeddings[start:end]
        @ jina_resume_embeddings.T
    )

    for local_i, row_idx in enumerate(range(start, end)):

        job_id = jobs_df.iloc[row_idx]["job_id"]

        relevant_ids = set(
            relevant_map[job_id]
        )

        job_scores = scores[local_i]

        # Only sort the top 100
        top_indices = np.argpartition(
            -job_scores, 99
        )[:100]

        top_indices = top_indices[
            np.argsort(-job_scores[top_indices])
        ]

        ranked_ids = [
            resume_ids[i]
            for i in top_indices
        ]

        # Recall@K
        for k in ks:
            hits = len(
                set(ranked_ids[:k]) & relevant_ids
            )

            metrics[f"recall@{k}"].append(
                hits / len(relevant_ids)
            )

        # MRR
        rr = 0.0

        for rank, resume_id in enumerate(
            ranked_ids,
            start=1
        ):
            if resume_id in relevant_ids:
                rr = 1.0 / rank
                break

        metrics["mrr"].append(rr)

        # nDCG
        for k in [5, 10]:

            dcg = sum(
                1.0 / np.log2(i + 2)
                for i, resume_id
                in enumerate(ranked_ids[:k])
                if resume_id in relevant_ids
            )

            ideal_hits = min(
                len(relevant_ids),
                k
            )

            idcg = sum(
                1.0 / np.log2(i + 2)
                for i in range(ideal_hits)
            )

            metrics[f"ndcg@{k}"].append(
                dcg / idcg if idcg > 0 else 0.0
            )

    if end % 500 == 0 or end == len(jobs_df):
        print(
            f"Processed {end}/{len(jobs_df)} jobs"
        )

jina_metrics = {
    metric: np.mean(values)
    for metric, values in metrics.items()
}

print("\nJina Embeddings v3 Retrieval Results")
print("-" * 50)

for metric, value in jina_metrics.items():
    print(f"{metric.upper():<12}: {value:.6f}")

## Jina Embeddings v3

Jina Embeddings v3 was evaluated using its asymmetric retrieval setup:
`retrieval_query` for job descriptions and `retrieval_passage` for resumes.

The model produced 1024-dimensional embeddings and was evaluated using
Recall@1/3/5/10/25/50/100, MRR, and nDCG@5/10.

Jina v3 achieved Recall@100 of 0.082053, below the Skill-Aware BM25
candidate-pool Recall@100 of 0.105653 on the evaluated benchmark.

Therefore, Jina v3 was retained as a semantic retrieval baseline but was
not selected as the primary candidate-generation method.

In [ ]:
# Sanity-check the ground truth and candidate counts

print("Number of jobs:", len(jobs_df))
print("Number of resumes:", len(resumes_df))

match_counts = matches_df["relevant_resume_ids"].apply(len)

print("\nRelevant candidates per job:")
print(match_counts.describe())

print("\nUnique relevant counts:")
print(match_counts.value_counts().sort_index())

print("\nExample:")
print(matches_df.iloc[0])

In [ ]:
import numpy as np

# Exact must-have skill overlap baseline

def normalize_skill(skill):
    return str(skill).strip().lower()

resume_skill_sets = {
    row["resume_id"]: {
        normalize_skill(skill)
        for skill in row["skills"]
    }
    for _, row in resumes_df.iterrows()
}

skill_overlap_results = {
    "recall@1": [],
    "recall@3": [],
    "recall@5": [],
    "recall@10": [],
    "recall@25": [],
    "recall@50": [],
    "recall@100": [],
    "mrr": [],
    "ndcg@5": [],
    "ndcg@10": [],
}

for _, job in jobs_df.iterrows():

    job_id = job["job_id"]

    must_have = {
        normalize_skill(skill)
        for skill in job["must_have_skills"]
    }

    scored_resumes = []

    for resume_id, resume_skills in resume_skill_sets.items():

        overlap = len(
            must_have & resume_skills
        )

        overlap_ratio = (
            overlap / len(must_have)
            if must_have else 0.0
        )

        scored_resumes.append(
            (resume_id, overlap_ratio)
        )

    scored_resumes.sort(
        key=lambda x: x[1],
        reverse=True
    )

    ranked_ids = [
        resume_id
        for resume_id, _ in scored_resumes
    ]

    relevant_ids = set(
        matches_df[
            matches_df["job_id"] == job_id
        ].iloc[0]["relevant_resume_ids"]
    )

    for k in [1, 3, 5, 10, 25, 50, 100]:

        hits = len(
            set(ranked_ids[:k]) & relevant_ids
        )

        skill_overlap_results[
            f"recall@{k}"
        ].append(
            hits / len(relevant_ids)
        )

    # MRR
    rr = 0.0

    for rank, resume_id in enumerate(
        ranked_ids,
        start=1
    ):
        if resume_id in relevant_ids:
            rr = 1.0 / rank
            break

    skill_overlap_results["mrr"].append(rr)

    # nDCG
    for k in [5, 10]:

        dcg = sum(
            1.0 / np.log2(i + 2)
            for i, resume_id in enumerate(
                ranked_ids[:k]
            )
            if resume_id in relevant_ids
        )

        ideal_hits = min(
            len(relevant_ids),
            k
        )

        idcg = sum(
            1.0 / np.log2(i + 2)
            for i in range(ideal_hits)
        )

        skill_overlap_results[
            f"ndcg@{k}"
        ].append(
            dcg / idcg
            if idcg > 0 else 0.0
        )

skill_overlap_metrics = {
    metric: np.mean(values)
    for metric, values
    in skill_overlap_results.items()
}

print("Exact Skill-Overlap Baseline")
print("-" * 45)

for metric, value in skill_overlap_metrics.items():
    print(
        f"{metric.upper():<12}: {value:.6f}"
    )

In [ ]:
# Inspect exact skill-overlap ties for one job

job = jobs_df.iloc[0]

must_have = {
    normalize_skill(skill)
    for skill in job["must_have_skills"]
}

overlap_scores = []

for resume_id, resume_skills in resume_skill_sets.items():
    overlap = len(must_have & resume_skills)
    ratio = overlap / len(must_have) if must_have else 0.0
    overlap_scores.append(ratio)

overlap_scores = np.array(overlap_scores)

unique_scores, counts = np.unique(
    overlap_scores,
    return_counts=True
)

print("Job:", job["job_id"])
print("Must-have skills:", job["must_have_skills"])
print("Number of must-have skills:", len(must_have))
print()
print("Unique overlap scores:", len(unique_scores))
print()

for score, count in zip(unique_scores, counts):
    print(f"{score:.2f} -> {count} resumes")